# Functions

# Copy sprites.anim File Add new Frame Function

In [16]:
import os
import struct
import shutil
import binascii

def modify_sprites_anim(input_file, output_file, source_frame=511, target_position=530):
    """
    Modify sprites.anim file by copying a frame and adding it after a specific position
    
    Args:
        input_file: Path to the source sprites.anim file
        output_file: Path to the destination sprites.anim file
        source_frame: The frame to copy (index is 0-based)
        target_position: The position after which to insert the copied frame (0-based)
    """
    # Make a copy of the original file
    shutil.copy2(input_file, output_file)
    print(f"Created copy at {output_file}")
    
    # Open the input file in binary mode for reading
    with open(input_file, 'rb') as f:
        # Read the entire file
        file_data = f.read()
        
        # Verify signature is 'AA' (0x41 0x41)
        if file_data[:2] != b'\x41\x41':
            raise ValueError("Invalid file signature. Expected 'AA'")
        
        # Read current frame count from header
        current_frames = int.from_bytes(file_data[2:4], byteorder='big')
        print(f"Current frames: 0x{current_frames:04X} ({current_frames})")
        
        # Find all frame start positions
        frame_starts = []
        pos = 0
        while True:
            pos = file_data.find(b'\x53\x53', pos)
            if pos == -1:
                break
            frame_starts.append(pos)
            pos += 2
        
        print(f"Found {len(frame_starts)} frame start markers")
        
        # Validate the source and target frames
        if source_frame >= len(frame_starts):
            raise ValueError(f"Source frame {source_frame} exceeds total frames {len(frame_starts)}")
        
        if target_position >= len(frame_starts):
            raise ValueError(f"Target position {target_position} exceeds total frames {len(frame_starts)}")
        
        # Get the frame to copy
        source_frame_data = find_frame(file_data, source_frame, frame_starts)
        print(f"Source frame {source_frame} size: {len(source_frame_data)} bytes")
        print(f"First 16 bytes: {binascii.hexlify(source_frame_data[:16])}")
        
        # Get the position where we need to insert the new frame
        # If target_position is the last frame, we need to handle it differently
        is_last_frame = (target_position == current_frames - 1)
        
        if is_last_frame:
            # Find the end marker after the last frame
            insert_pos = file_data.find(b'\x43\x43', frame_starts[target_position])
            if insert_pos == -1:
                raise ValueError("Could not find end marker after the last frame")
            
            # Tileset data starts 2 bytes after the end marker
            tileset_data_offset = insert_pos + 2
            tileset_and_palette = file_data[tileset_data_offset:]
        else:
            # Insert position is the start of the next frame
            insert_pos = frame_starts[target_position + 1]
            # No tileset data to worry about
            tileset_and_palette = None
    
    # Now open the copy for writing
    with open(output_file, 'r+b') as f:
        # Update frame count in header to current + 1
        new_frames = current_frames + 1
        f.seek(2)
        f.write(new_frames.to_bytes(2, byteorder='big'))
        print(f"Updated frame count to: 0x{new_frames:04X} ({new_frames})")
        
        if is_last_frame:
            # If we're inserting after the last frame, we need to:
            # 1. Position at the end marker of the last frame
            # 2. Replace it with our new frame
            # 3. Write a new end marker
            # 4. Write the tileset and palette data
            
            f.seek(insert_pos)
            # Write the new frame (which already has its 0x53 0x53 start marker)
            f.write(source_frame_data)
            # Write the end marker for the new last frame
            f.write(b'\x43\x43')
            # Write the tileset and palette data
            f.write(tileset_and_palette)
        else:
            # If we're inserting in the middle:
            # 1. Read all data from the insertion point to the end
            # 2. Position at the insertion point
            # 3. Write our new frame
            # 4. Write back the remaining data
            
            f.seek(insert_pos)
            remaining_data = f.read()
            
            f.seek(insert_pos)
            f.write(source_frame_data)
            f.write(remaining_data)
    
    # Verify the modification
    with open(output_file, 'rb') as f:
        verify_data = f.read()
        
        # Check frame count was updated
        new_count = int.from_bytes(verify_data[2:4], byteorder='big')
        print(f"Verified new frame count: 0x{new_count:04X}")
        
        # Count frame signatures
        frame_count = 0
        pos = 0
        while True:
            pos = verify_data.find(b'\x53\x53', pos)
            if pos == -1:
                break
            frame_count += 1
            pos += 2
        
        print(f"Total frames found after modification: {frame_count}")
        
        # Quick summary of file size changes
        original_size = len(file_data)
        new_size = len(verify_data)
        print(f"Original file size: {original_size} bytes")
        print(f"New file size: {new_size} bytes")
        print(f"Difference: {new_size - original_size} bytes")

def find_frame(data, frame_index, frame_starts=None):
    """Find and extract the specified frame from the file data"""
    if frame_starts is None:
        # Find all frame start markers (0x53 0x53)
        frame_starts = []
        pos = 0
        while True:
            pos = data.find(b'\x53\x53', pos)
            if pos == -1:
                break
            frame_starts.append(pos)
            pos += 2
    
    total_frames = len(frame_starts)
    print(f"Total frames found: {total_frames}")
    
    if frame_index >= total_frames:
        print(f"Warning: Requested frame {frame_index} exceeds total frames {total_frames}")
        return None
    
    start_pos = frame_starts[frame_index]
    
    # If it's not the last frame, the end is the start of the next frame
    if frame_index < total_frames - 1:
        end_pos = frame_starts[frame_index + 1]
        # Extract the frame data including the start marker
        frame_data = data[start_pos:end_pos]
    else:
        # For the last frame, find the end marker 0x43 0x43 after this frame
        # The last frame is special - it's the only one with an end marker
        end_marker_pos = data.find(b'\x43\x43', start_pos)
        if end_marker_pos == -1:
            print(f"Warning: Could not find end marker for the last frame")
            return None
        # Extract the frame data including the start marker but NOT including the end marker
        frame_data = data[start_pos:end_marker_pos]
    
    print(f"Found frame {frame_index} at offset 0x{start_pos:X}, size {len(frame_data)} bytes")
    
    return frame_data

def inspect_file(filepath):
    """Inspect the file to verify frame markers and structure"""
    with open(filepath, 'rb') as f:
        data = f.read()
        
        # Check header
        if data[:2] != b'\x41\x41':
            print("WARNING: Invalid file signature!")
            
        # Check frame count in header
        frame_count = int.from_bytes(data[2:4], byteorder='big')
        
        # Find all frame markers
        frame_markers = []
        pos = 0
        while True:
            pos = data.find(b'\x53\x53', pos)
            if pos == -1:
                break
            frame_markers.append(pos)
            pos += 2
        
        # Find the end marker (should only be one for the last frame)
        end_marker_pos = data.rfind(b'\x43\x43')
            
        print(f"File inspection of {filepath}:")
        print(f"  Size: {len(data)} bytes")
        print(f"  Frame count in header: 0x{frame_count:04X} ({frame_count})")
        print(f"  Frame markers found: {len(frame_markers)}")
        print(f"  End marker position: 0x{end_marker_pos:X}")
        
        # Show frames around our target position
        target_idx = 534
        start_idx = max(0, target_idx - 2)
        end_idx = min(len(frame_markers), target_idx + 4)
        
        print(f"Frames around target position (534):")
        for i in range(start_idx, end_idx):
            pos = frame_markers[i]
            next_pos = frame_markers[i+1] if i+1 < len(frame_markers) else end_marker_pos
            size = next_pos - pos
            print(f"  Frame {i}: offset 0x{pos:X}, size {size} bytes")
        
        # Verify frame count vs actual frames
        if len(frame_markers) != frame_count:
            print(f"  WARNING: Frame count in header ({frame_count}) does not match actual frame count ({len(frame_markers)})")

if __name__ == "__main__":
    src_file = "../src/graphics/sprites.anim"
    dst_file = "../src/graphics/sprites_new.anim"
    
    # Inspect the original file first
    print("Inspecting original file:")
    inspect_file(src_file)
    
    # Modify the file, creating a copy
    print("\nModifying file:")
    modify_sprites_anim(src_file, dst_file, source_frame=511, target_position=529)
    
    # Inspect the modified file
    print("\nInspecting modified file:")
    inspect_file(dst_file)
    
    print(f"\nSuccessfully created modified copy at {dst_file}")

Inspecting original file:
File inspection of ../src/graphics/sprites.anim:
  Size: 215934 bytes
  Frame count in header: 0x0224 (548)
  Frame markers found: 549
  End marker position: 0x20348
Frames around target position (534):
  Frame 532: offset 0x85A6, size 46 bytes
  Frame 533: offset 0x85D4, size 46 bytes
  Frame 534: offset 0x8602, size 46 bytes
  Frame 535: offset 0x8630, size 46 bytes
  Frame 536: offset 0x865E, size 54 bytes
  Frame 537: offset 0x8694, size 54 bytes

Modifying file:
Created copy at ../src/graphics/sprites_new.anim
Current frames: 0x0224 (548)
Found 549 frame start markers
Total frames found: 549
Found frame 511 at offset 0x81E0, size 46 bytes
Source frame 511 size: 46 bytes
First 16 bytes: b'5353004d003b00400030000000000000'
Updated frame count to: 0x0225 (549)
Verified new frame count: 0x0225
Total frames found after modification: 550
Original file size: 215934 bytes
New file size: 215980 bytes
Difference: 46 bytes

Inspecting modified file:
File inspection 